# SQL Teil 4: Live-Betrieb

**Voraussetzung:** [Teil 3: Abfragen](../3-abfragen/abfragen.ipynb)
**Worum es geht:** [Anforderungen.md](Anforderungen.md)

Das Spiel ist online. 5.000 Accounts, 15.000 Charaktere, und jeden Tag kommen Tickets rein. Jedes Ticket beschreibt ein **echtes Problem**. Ihr stellt es zuerst nach, und erst dann kommt das Werkzeug, das es löst.

| Ticket | Problem | Werkzeug |
|---|---|---|
| 1 | Login-Historie lädt ewig | Index |
| 2 | Doppelte Gilden, zwei Gildenleitungen | UNIQUE, weitergedacht |
| 3 | Beim Handeln entsteht Gold aus dem Nichts | Transaktion, ACID |
| 4 | Support soll keine Passwort-Hashes sehen | View, Rechte (DCL) |
| 5 | Wer hat sich eine Million Gold gegeben? | Trigger |
| 6 | Login ohne Passwort | SQL-Injection, Parameter |
| 7 | Spieler*in verlangt Löschung der eigenen Daten | DSGVO, Löschkonzept |
| 8 | Alle Gilden versehentlich gelöscht | Backup und Wiederherstellung |

**Gewollte Fehlermeldungen:** In einigen Übungen **soll** eine rote Fehlermeldung kommen. Das steht jeweils dabei. **Run All** stoppt beim ersten Fehler, führt diese Zellen deshalb einzeln aus.

# Setup

Ihr arbeitet auf einer eigenen Kopie der Referenzdatenbank, `live.db`. Dazu kommt eine Tabelle `login` mit **500.000** Logins (siehe [login.sql](login.sql)).

- Beim ersten Mal wird eine Vorlage `vorlage.db` gebaut. Das dauert ein paar Sekunden.
- **Jedes Mal**, wenn ihr diese Zelle ausführt, wird `live.db` auf den Stand der Vorlage **zurückgesetzt**. Alles, was ihr geändert habt, ist dann weg. So könnt ihr jederzeit neu anfangen.
- Am besten vorher **Restart**, damit keine alte Verbindung mehr offen ist.

Außerdem gibt es die Hilfsfunktion `messen(sql)`. Sie führt eine Abfrage 20-mal aus und gibt die durchschnittliche Dauer aus.

In [ ]:
import sqlite3
import time
from pathlib import Path

VORLAGE = Path("vorlage.db")
DB = Path("live.db")

if not VORLAGE.exists():
    print("Baue die Vorlage, einen Moment ...")
    tmp = Path("vorlage.tmp")
    tmp.unlink(missing_ok=True)
    con = sqlite3.connect(tmp)
    con.executescript(Path("../referenz/setup.sql").read_text(encoding="utf-8"))
    con.executescript(Path("login.sql").read_text(encoding="utf-8"))
    con.close()
    tmp.rename(VORLAGE)

# live.db auf den Stand der Vorlage zurücksetzen
quelle = sqlite3.connect(VORLAGE)
ziel = sqlite3.connect(DB)
quelle.backup(ziel)
quelle.close()
ziel.close()
print("live.db ist bereit:", DB.resolve())


def messen(sql, wiederholungen=20):
    """Führt eine Abfrage mehrmals aus und gibt die durchschnittliche Dauer aus."""
    con = sqlite3.connect(DB)
    start = time.perf_counter()
    for _ in range(wiederholungen):
        con.execute(sql).fetchall()
    dauer = (time.perf_counter() - start) / wiederholungen * 1000
    con.close()
    print(f"{dauer:.2f} ms pro Abfrage (Durchschnitt aus {wiederholungen} Läufen)")

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 20
%sql sqlite:///live.db
%sql PRAGMA foreign_keys = ON;

# Ticket 1: „Die Login-Historie lädt ewig“

Der Support schaut sich bei Problemen die letzten Logins eines Accounts an. Das Support-Tool führt diese Abfrage aus:

In [ ]:
%%sql
SELECT zeitpunkt, ip
FROM login
WHERE account_id = 4711
ORDER BY zeitpunkt DESC
LIMIT 10;

Das Ergebnis kommt schnell genug, oder? Messt genauer:

In [ ]:
messen("SELECT zeitpunkt, ip FROM login WHERE account_id = 4711 ORDER BY zeitpunkt DESC LIMIT 10")

Das klingt nach wenig. Aber 50 Support-Leute klicken den ganzen Tag, und die Tabelle wächst jeden Tag um viele tausend Zeilen.

### Was macht die Datenbank da?
`EXPLAIN QUERY PLAN` vor einer Abfrage zeigt, **wie** die Datenbank sie ausführen will, ohne sie auszuführen.

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT zeitpunkt, ip
FROM login
WHERE account_id = 4711
ORDER BY zeitpunkt DESC
LIMIT 10;

`SCAN login` heißt: Die Datenbank liest **alle 500.000 Zeilen** und prüft bei jeder, ob `account_id = 4711` ist. Gefunden werden am Ende 100 Zeilen.

Stellt euch ein Telefonbuch vor, das nicht alphabetisch sortiert ist. Um „Müller“ zu finden, müsstet ihr jede Seite lesen. Ein **Index** ist wie ein sortiertes Register: Die Datenbank springt direkt an die richtige Stelle.

### Übung 1.1
Legt einen Index auf die Spalte `account_id` an:

```sql
CREATE INDEX idx_login_account ON login(account_id);
```

Führt danach `EXPLAIN QUERY PLAN` und `messen(...)` von oben noch einmal aus. Was steht jetzt im Plan? Um welchen Faktor ist die Abfrage schneller?

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Index anlegen' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: EXPLAIN QUERY PLAN ...' AS aufgabe;

In [ ]:
# TODO: messen(...) erneut aufrufen

### Übung 1.2: Noch eine Stufe besser ⭐⭐⭐
Im Plan steht vermutlich noch `USE TEMP B-TREE FOR ORDER BY`. Die Datenbank holt die 100 Zeilen und muss sie danach **sortieren**.

Ein Index kann auch über **mehrere Spalten** gehen, z. B. `(account_id, zeitpunkt)`. Dann liegen die Logins eines Accounts schon nach Zeit sortiert im Index.

Legt so einen Index an und schaut euch den Plan an. Was fällt weg?

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 1.3: Wann hilft ein Index nicht?
Für die Spalte `email` in `account` gibt es schon einen Index, weil sie `UNIQUE` ist. Vergleicht die Pläne dieser beiden Abfragen:

```sql
SELECT * FROM account WHERE email = 'player42@example.com';
SELECT * FROM account WHERE LOWER(email) = 'player42@example.com';
```

Warum wird der Index bei der zweiten nicht genutzt?

*Tipp: Das Register im Telefonbuch ist nach dem Namen sortiert, nicht nach dem Namen in Kleinbuchstaben.*

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 1.4: Woher kommt der Index auf `email`?
Diese Abfrage zeigt alle Indexe der Datenbank. Den mit `idx_` habt ihr angelegt. Woher kommen die anderen?

In [ ]:
%%sql
SELECT name, tbl_name FROM sqlite_schema WHERE type = 'index' ORDER BY tbl_name;

*Antwort:* ...

<details>
<summary>Auflösung</summary>

`UNIQUE` und zusammengesetzte Primärschlüssel legen automatisch einen Index an (`sqlite_autoindex_...`). Sonst müsste die Datenbank bei jedem `INSERT` alle Zeilen durchsuchen, um zu prüfen, ob es den Wert schon gibt. Ein einspaltiger `INTEGER PRIMARY KEY` braucht keinen eigenen Index: In SQLite ist er die Zeilennummer selbst, nach der die Tabelle sowieso sortiert gespeichert ist.

</details>

### Merke: Was kostet ein Index?

Ein Index ist nicht kostenlos:

- Er braucht **Speicherplatz**. Prüft es: Wie groß ist `live.db` jetzt im Vergleich zu `vorlage.db`? (Rechtsklick auf die Datei, oder `DB.stat().st_size` in Python.)
- Bei jedem `INSERT`, `UPDATE` und `DELETE` muss die Datenbank **jeden** Index der Tabelle mit aktualisieren. Schreiben wird also langsamer.

Faustregel: Einen Index legt man auf Spalten, nach denen **oft gesucht, gefiltert, verbunden oder sortiert** wird, vor allem bei großen Tabellen. Typische Kandidaten sind **Fremdschlüsselspalten**: Die bekommen in SQLite nicht automatisch einen Index. Einen Index auf jede Spalte legt man nicht.

# Ticket 2: „Zwei Gilden mit fast demselben Namen“

Jemand hat eine Gilde gegründet, die genauso heißt wie eine bekannte, nur kleingeschrieben. Die App hat das zugelassen:

In [ ]:
%%sql
INSERT INTO guild (name, description)
VALUES ('iron vanguard', 'Die echte Iron Vanguard. Ehrlich.');

SELECT id, name FROM guild WHERE LOWER(name) = 'iron vanguard';

`name` ist doch `UNIQUE`. Warum ging das durch?

`UNIQUE` vergleicht Texte **genau**, also mit Groß- und Kleinschreibung. Für die Datenbank sind `Iron Vanguard` und `iron vanguard` zwei verschiedene Werte.

### Übung 2.1
Ein **eindeutiger Index** (`CREATE UNIQUE INDEX`) wirkt wie `UNIQUE`, kann aber mehr. Mit `COLLATE NOCASE` vergleicht er ohne Rücksicht auf Groß- und Kleinschreibung:

```sql
CREATE UNIQUE INDEX ux_guild_name_nocase ON guild(name COLLATE NOCASE);
```

1. Führt die Anweisung aus. **Eine rote Fehlermeldung ist hier gewollt.** Warum lässt sich der Index nicht anlegen?
2. Behebt die Ursache und legt den Index dann an.
3. Versucht noch einmal, `iron vanguard` einzufügen. Jetzt muss es scheitern.

*Gut zu wissen:* `NOCASE` kennt in SQLite nur die Buchstaben A bis Z. `Ä` und `ä` gelten weiterhin als verschieden.

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

## Ticket 2b: „Unsere Gilde hat zwei Anführer“

Durch einen Bug in der App wurde ein normales Mitglied von *Order of the Dawn* (`guild_id = 1`) zum Guildwarden (`role_id = 1`) befördert, obwohl es schon einen gibt:

In [ ]:
%%sql
UPDATE player_guild_role SET role_id = 1
WHERE player_id = 47 AND guild_id = 1;

SELECT guild_id, COUNT(*) AS guildwardens
FROM player_guild_role
WHERE role_id = 1
GROUP BY guild_id;

Die Regel „genau **ein** Guildwarden pro Gilde“ steht in den Anforderungen, aber nicht in der Datenbank. `UNIQUE(guild_id)` geht nicht, denn jede Gilde hat natürlich viele Mitglieder.

### Übung 2.2
Ein **partieller Index** gilt nur für die Zeilen, die eine Bedingung erfüllen:

```sql
CREATE UNIQUE INDEX name ON tabelle(spalte) WHERE bedingung;
```

1. Baut einen eindeutigen Index, der dafür sorgt, dass es pro `guild_id` höchstens **eine** Zeile mit `role_id = 1` gibt.
2. Er lässt sich erst anlegen, wenn der Fehler oben behoben ist. Setzt Charakter 47 wieder auf `role_id = 4` (Member).
3. Testet: Ein zweiter Guildwarden muss jetzt abgelehnt werden.

**Erwartetes Ergebnis:** Die Abfrage oben zeigt danach für jede Gilde wieder genau 1.

**Frage:** Der Index sichert ab, dass es **höchstens** einen Guildwarden gibt. Was ist mit **mindestens** einem? Was passiert, wenn der einzige Guildwarden die Gilde verlässt?

*Antwort:* ...

<details>
<summary>Auflösung</summary>

Das kann kein Index absichern. Ein Index verbietet doppelte Werte, er kann aber nicht verlangen, dass ein Wert vorhanden ist. Dafür braucht es z. B. einen Trigger, der das Löschen des letzten Guildwardens verhindert, oder eine Regel im Programm.

</details>

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Merke: Eindeutigkeit

| Regel | Umsetzung |
|---|---|
| Ein Wert darf nur einmal vorkommen | `UNIQUE` an der Spalte |
| Eine **Kombination** darf nur einmal vorkommen | `UNIQUE (spalte_a, spalte_b)` am Ende von `CREATE TABLE` |
| Eindeutig ohne Groß- und Kleinschreibung | `CREATE UNIQUE INDEX ... (spalte COLLATE NOCASE)` |
| Eindeutig nur für bestimmte Zeilen | `CREATE UNIQUE INDEX ... WHERE bedingung` (partieller Index) |

Jede dieser Regeln steht **in der Datenbank**. Egal, welche App oder welches Skript schreibt: Die Regel gilt immer.

# Ticket 3: „Beim Handeln entsteht Gold aus dem Nichts“

Charakter 15 kauft von Charakter 1 ein Schwert für **500 Gold**. Charakter 15 hat aber nur **335 Gold**.

Die App bucht so:

```sql
UPDATE player SET gold = gold + 500 WHERE id = 1;   -- Verkäufer bekommt Gold
UPDATE player SET gold = gold - 500 WHERE id = 15;  -- Käufer zahlt
```

### Übung 3.1
1. Fragt das **gesamte Gold aller Charaktere** ab und merkt euch die Zahl.
2. Führt die beiden Anweisungen der App in **einer** Zelle aus. **Eine rote Fehlermeldung ist hier gewollt.** Welche Regel schlägt zu?
3. Fragt das gesamte Gold noch einmal ab. Was ist passiert?

**Erwartetes Ergebnis:** vorher 62603537, nachher 62604037

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: gesamtes Gold abfragen' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: die beiden UPDATEs der App' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: gesamtes Gold noch einmal abfragen' AS aufgabe;

Die erste Anweisung hat geklappt, die zweite nicht. Charakter 1 hat 500 Gold bekommen, die niemand bezahlt hat. In einem echten Spiel würde das jemand ausnutzen und sich auf diese Weise beliebig viel Gold beschaffen.

### Die Lösung: Transaktion
Eine **Transaktion** fasst mehrere Anweisungen zusammen: **Entweder alle oder keine.**

```sql
BEGIN;                -- Transaktion starten
UPDATE ...;
UPDATE ...;
COMMIT;               -- alles klappt: Änderungen dauerhaft speichern
```

Geht unterwegs etwas schief, nimmt `ROLLBACK;` **alle** Änderungen seit `BEGIN` zurück.

### Übung 3.2
1. Macht den Schaden rückgängig: Zieht Charakter 1 die 500 Gold wieder ab.
2. Führt den Handel noch einmal aus, diesmal mit `BEGIN` davor. Der Fehler kommt wieder (gewollt). Die Transaktion ist jetzt noch offen.
3. Führt in einer eigenen Zelle `ROLLBACK;` aus.
4. Prüft das gesamte Gold: Es muss wieder 62603537 sein.
5. Jetzt ein Handel, der klappen soll: Charakter 15 kauft einen Heiltrank für 100 Gold. Diesmal mit `BEGIN` und `COMMIT`. Das gesamte Gold darf sich nicht ändern.

In [ ]:
%%sql
SELECT 'TODO: Schaden rückgängig machen' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: BEGIN; UPDATE ...; UPDATE ...;' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: ROLLBACK' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: gesamtes Gold prüfen' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Handel mit BEGIN und COMMIT' AS aufgabe;

### In einem Programm
Im Notebook entscheidet ihr selbst, ob ihr `COMMIT` oder `ROLLBACK` ausführt. Ein Programm macht das automatisch. In Python mit `sqlite3` zum Beispiel so:

```python
with con:   # bei Erfolg COMMIT, bei einem Fehler ROLLBACK
    con.execute("UPDATE player SET gold = gold + 500 WHERE id = 1")
    con.execute("UPDATE player SET gold = gold - 500 WHERE id = 15")
```

### ACID: Was eine Transaktion garantiert

| Buchstabe | Bedeutung | Im Beispiel |
|---|---|---|
| **A**tomicity (Atomarität) | Alles oder nichts. | Entweder bekommt der Verkäufer das Gold **und** der Käufer zahlt, oder keins von beiden. |
| **C**onsistency (Konsistenz) | Vorher und nachher sind die Daten stimmig. Die Datenbank prüft dabei alle Constraints. Fachliche Regeln sichert das Programm, indem es zusammengehörige Schritte in eine Transaktion packt. | Das Gold aller Charaktere zusammen ist nach jedem Handel gleich. Es entsteht keins aus dem Nichts. |
| **I**solation | Gleichzeitige Transaktionen stören sich nicht. | Zwei Käufe beim selben Charakter zur selben Zeit sehen keine halben Zwischenstände. |
| **D**urability (Dauerhaftigkeit) | Nach `COMMIT` ist die Änderung gespeichert, auch wenn danach der Server abstürzt. | Der Handel ist auch nach einem Stromausfall noch da. |

**Frage:** Wo in eurem Alltag (Bank, Onlineshop, Lager) wäre ein halber Vorgang ein Problem?

*Antwort:* ...

# Ticket 4: „Der Support soll keine Passwort-Hashes sehen“

Das Support-Team braucht Infos zu Accounts: E-Mail, Anmeldedaten, Anzahl Charaktere. Bisher arbeitet es direkt auf der Tabelle `account`. Schaut euch an, was es dabei sieht:

In [ ]:
%%sql
SELECT * FROM account LIMIT 5;

Neben E-Mail und Anmeldedaten steht dort auch `password_hash`. Den braucht der Support **nicht**. Selbst ein Hash ist gefährlich: Wer ihn kennt, kann auf dem eigenen Rechner beliebig viele Passwörter durchprobieren, bis eins passt.

Eine **View** ist eine gespeicherte Abfrage, die man wie eine Tabelle benutzen kann. Sie speichert **keine eigenen Daten**. Bei jedem Zugriff wird die Abfrage dahinter neu ausgeführt.

```sql
CREATE VIEW name AS
SELECT ...;
```

### Übung 4.1
Legt eine View `support_account` an mit den Spalten `id`, `email`, `created_at`, `last_login` und `charaktere` (Anzahl der Charaktere). Accounts ohne Charakter sollen mit 0 erscheinen.

Fragt danach die View ab, als wäre sie eine Tabelle: Wie viele Charaktere hat `player42@example.com`?

**Erwartetes Ergebnis:**

| id | email | charaktere |
|---|---|---|
| 42 | player42@example.com | 1 |

In [ ]:
%%sql
SELECT 'TODO: CREATE VIEW support_account AS ...' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 4.2
Legt für Account 42 einen neuen Charakter an (Klasse 1, Fraktion 1, Benutzername frei wählbar) und fragt die View erneut ab. Musstet ihr an der View etwas ändern?

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 4.3: Views gegen lange Joins ⭐⭐
Views sind nicht nur für den Datenschutz gut. Sie verstecken auch komplizierte Joins.

Legt eine View `charakter_uebersicht` an mit Benutzername, Name der Klasse, Name der Fraktion, aktuellem Level und Gold. Beantwortet damit: Wie viele Assassins auf Level 5 gibt es?

**Erwartetes Ergebnis:** 153

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Und wer darf was? Rechte mit DCL

Eine View allein schützt in SQLite **nichts**: Wer die Datei `live.db` öffnen kann, sieht alle Tabellen. SQLite hat keine Benutzerverwaltung.

Server-Datenbanken wie PostgreSQL, MySQL oder Microsoft SQL Server haben eine. Dort vergibt man Rechte mit der **Data Control Language (DCL)**:

```sql
-- PostgreSQL, funktioniert NICHT in SQLite
CREATE ROLE support LOGIN PASSWORD 'geheim';

GRANT SELECT ON support_account TO support;     -- darf die View lesen
-- auf die Tabelle account selbst bekommt "support" kein Recht

REVOKE SELECT ON support_account FROM support;  -- Recht wieder entziehen
```

Die Rolle `support` kann dann die View lesen, aber nicht direkt die Tabelle `account`. So kommt sie nie an die Passwort-Hashes.

Die SQL-Befehle teilt man in Gruppen ein:

| Gruppe | Bedeutung | Befehle |
|---|---|---|
| **DDL** | Data Definition Language: Struktur anlegen und ändern | `CREATE`, `ALTER`, `DROP` |
| **DML** | Data Manipulation Language: Daten ändern | `INSERT`, `UPDATE`, `DELETE` |
| **DQL** | Data Query Language: Daten abfragen | `SELECT` |
| **DCL** | Data Control Language: Rechte vergeben | `GRANT`, `REVOKE` |
| **TCL** | Transaction Control Language: Transaktionen steuern | `BEGIN`, `COMMIT`, `ROLLBACK` |

Manche Bücher zählen `SELECT` zur DML oder führen TCL nicht als eigene Gruppe.

**Prinzip der minimalen Rechte:** Jede Rolle bekommt nur die Rechte, die sie für ihre Arbeit wirklich braucht. Auch die App selbst sollte sich nicht als Administrator mit der Datenbank verbinden.

# Ticket 5: „Wer hat sich eine Million Gold gegeben?“

Die nächste Zelle stellt nach, was passiert ist: Charakter 40 hat plötzlich über eine Million Gold.

In [ ]:
%%sql
UPDATE player SET gold = gold + 1000000 WHERE id = 40;

SELECT id, username, gold FROM player WHERE id = 40;

### Übung 5.0
Findet heraus, **wann** und **wodurch** das Gold dazugekommen ist. Schaut euch dafür ruhig alle Tabellen an.

*Antwort:* ...

Ihr werdet nichts finden. Die Datenbank speichert nur den **aktuellen** Stand, nicht, wie er entstanden ist. Das Studio will deshalb künftig **jede** Änderung am Gold protokollieren, egal ob sie aus der App, einem Admin-Tool oder einem Skript kommt.

Ein **Trigger** ist SQL-Code, den die Datenbank **automatisch** ausführt, wenn etwas passiert: beim `INSERT`, `UPDATE` oder `DELETE` einer Tabelle. Im Trigger gibt es `OLD` (die Zeile vorher) und `NEW` (die Zeile nachher).

Zuerst die Tabelle für das Protokoll:

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS gold_log (
    id        INTEGER PRIMARY KEY AUTOINCREMENT,
    player_id INTEGER NOT NULL,
    alt       INTEGER NOT NULL,
    neu       INTEGER NOT NULL,
    geaendert DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP
);

### Übung 5.1
Schreibt einen Trigger `tr_gold_log`, der nach jeder Änderung von `gold` in `player` eine Zeile in `gold_log` schreibt. Gerüst:

```sql
CREATE TRIGGER tr_gold_log
AFTER UPDATE OF gold ON player
WHEN OLD.gold <> NEW.gold
BEGIN
    INSERT INTO gold_log (...) VALUES (...);
END;
```

Testet ihn: Gebt Charakter 40 noch einmal eine Million Gold dazu und schaut in `gold_log`.

**Erwartetes Ergebnis:**

| player_id | alt | neu |
|---|---|---|
| 40 | 1002275 | 2002275 |

In [ ]:
%%sql
SELECT 'TODO: CREATE TRIGGER ...' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Trigger testen' AS aufgabe;

### Übung 5.2: Trigger als Regel ⭐⭐⭐
Neue Regel vom Game-Design: Ein Charakter darf in **höchstens 2 Gilden** sein.

Das lässt sich nicht mit `CHECK` lösen, denn `CHECK` sieht nur die **eine** Zeile, die gerade eingefügt wird, nicht die anderen Zeilen der Tabelle. Ein Trigger kann die Tabelle abfragen und den Vorgang abbrechen:

```sql
CREATE TRIGGER tr_max_gilden
BEFORE INSERT ON player_guild
WHEN (SELECT COUNT(*) FROM ...) >= 2
BEGIN
    SELECT RAISE(ABORT, 'Ein Charakter darf in höchstens 2 Gilden sein.');
END;
```

Vervollständigt den Trigger. Testet ihn mit Charakter 43, der schon in 2 Gilden ist: Ein Beitritt zu Gilde 6 muss mit eurer Fehlermeldung scheitern (**gewollt**).

In [ ]:
%%sql
SELECT 'TODO: CREATE TRIGGER tr_max_gilden ...' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Trigger testen' AS aufgabe;

### Merke: Trigger mit Vorsicht
Trigger sind mächtig, aber **unsichtbar**. Wer nur `UPDATE player SET gold = ...` liest, sieht nicht, dass dabei noch eine andere Tabelle beschrieben wird. Bei vielen Triggern versteht irgendwann niemand mehr, was bei einer Änderung alles passiert.

Faustregel:

1. Was `NOT NULL`, `UNIQUE`, `CHECK` oder ein Fremdschlüssel kann, macht man damit.
2. Trigger nimmt man für Protokolle (wie `gold_log`) und für Regeln, die andere Zeilen oder Tabellen brauchen.
3. Jeder Trigger wird dokumentiert.

# Ticket 6: „Jemand hat sich ohne Passwort eingeloggt“

> **Wichtig:** Die folgenden Angriffe probiert ihr nur in diesem Notebook aus. An fremden Systemen ist so etwas verboten, auch „nur zum Testen“.

So prüft eine (schlecht programmierte) Login-Funktion E-Mail und Passwort. Die Testdaten speichern statt echter Hashes einfache Platzhalter wie `demo_2834`. Für das Ticket tun wir so, als wäre das das Passwort.

In [ ]:
def login(email, passwort):
    con = sqlite3.connect(DB)
    sql = f"SELECT id, email FROM account WHERE email = '{email}' AND password_hash = '{passwort}'"
    print("Ausgeführt wird:", sql)
    zeile = con.execute(sql).fetchone()
    con.close()
    return zeile


print(login("player42@example.com", "falsch"))

Falsches Passwort, also `None`: kein Login. So weit, so gut.

### Übung 6.1
Die Funktion setzt eure Eingabe **direkt** in den SQL-Text ein. Was passiert, wenn die Eingabe selbst SQL enthält?

1. Loggt euch als `player42@example.com` ein, **ohne** das Passwort zu kennen. *Tipp: `--` leitet in SQL einen Kommentar ein, alles danach wird ignoriert.*
2. Loggt euch ein, **ohne** eine E-Mail-Adresse zu kennen. *Tipp: `OR 1 = 1` ist immer wahr.*

Schaut euch bei jedem Versuch die Zeile „Ausgeführt wird“ an.

**Erwartetes Ergebnis:** Versuch 1 liefert `(42, 'player42@example.com')`. Versuch 2 liefert irgendeinen fremden Account, bei der Referenz `(1000, 'player1000@example.com')`.

In [ ]:
# TODO: login(...) mit präparierter E-Mail-Adresse aufrufen

In [ ]:
# TODO: login(...) ohne bekannte E-Mail-Adresse

Das ist **SQL-Injection**: Eingaben werden als SQL-Code ausgeführt statt als Daten behandelt. Sie gehört seit Jahren zu den häufigsten Sicherheitslücken in Webanwendungen.

### Die Lösung: Parameter
Statt die Eingabe in den Text zu kleben, schreibt man Platzhalter `?` und gibt die Werte **getrennt** mit. Die Datenbank behandelt sie dann immer als Daten, nie als Code. Das nennt man **parametrisierte Abfrage** oder **Prepared Statement**.

```python
con.execute("SELECT id FROM account WHERE email = ? AND password_hash = ?", (email, passwort))
```

### Übung 6.2
Schreibt eine Funktion `login_sicher` mit Parametern und wiederholt beide Angriffe. Beide müssen `None` liefern. Ein Login mit dem **richtigen** Passwort muss weiterhin klappen. (Den Platzhalter von Account 42 könnt ihr vorher per SQL nachschauen.)

In [ ]:
# TODO: def login_sicher(email, passwort): ...

### Merke
- **Nie** Eingaben mit `+`, f-Strings oder `format` in SQL einbauen. Immer Parameter.
- Anführungszeichen selbst „escapen“ (z. B. mit `replace`) ist keine Lösung. Es gibt zu viele Sonderfälle.
- Nicht nur Formulare sind Eingaben: auch URLs, Cookies, Dateinamen, Daten aus anderen Systemen.
- `sqlite3` führt mit `execute` nur **eine** Anweisung aus. Ein angehängtes `; DROP TABLE account` scheitert hier deshalb. Andere Treiber erlauben mehrere Anweisungen. Darauf verlassen darf man sich nie.
- Echte Apps vergleichen Passwörter übrigens nicht in SQL. Sie laden den Hash zur E-Mail-Adresse und prüfen ihn im Programm mit einem Verfahren wie bcrypt oder Argon2.

# Ticket 7: „Bitte löscht alle meine Daten“

Der Mensch hinter Account 42 schreibt: *„Ich höre auf zu spielen. Bitte löscht alle meine Daten.“*

Nach der **DSGVO** muss das Studio personenbezogene Daten löschen, wenn es keinen Grund mehr gibt, sie zu speichern. Personenbezogen ist alles, was sich einer Person zuordnen lässt: E-Mail-Adresse, Charaktere, Logins mit IP-Adressen, Protokolle.

### Übung 7.1
Löscht den Account mit `DELETE FROM account WHERE id = 42;`. **Eine rote Fehlermeldung ist hier gewollt.** Warum scheitert es, obwohl `player` doch `ON DELETE CASCADE` hat?

*Tipp: Welche Tabellen verweisen noch auf `account`? Schaut in [login.sql](login.sql).*

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 7.2
Findet **alle** Stellen, an denen noch Daten zu Account 42 stehen. Denkt auch an Tabellen **ohne** Fremdschlüssel, z. B. `gold_log` aus Ticket 5. Notiert für jede Tabelle, wie viele Zeilen betroffen sind.

**Erwartetes Ergebnis** (wenn ihr Ticket 4 bis 6 wie beschrieben gemacht habt): 100 Logins, 2 Charaktere, mindestens 1 Zeile in `gold_log`.

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 7.3
Schreibt ein **Löschskript**, das alles zu Account 42 entfernt, und zwar in einer **Transaktion**: Entweder ist danach alles weg oder nichts. Prüft danach, dass nichts übrig ist.

In [ ]:
%%sql
SELECT 'TODO: BEGIN; ... COMMIT;' AS aufgabe;

### Zum Nachdenken: Löschen oder anonymisieren?

Nicht alles darf sofort gelöscht werden. Rechnungen zum Beispiel müssen eine bestimmte Zeit **aufbewahrt** werden. Dann ist die Löschung erst nach Ablauf der Frist erlaubt, bis dahin werden die Daten gesperrt.

Manchmal will man Daten für Statistiken behalten, z. B. „Wie viele Logins gab es im März?“. Dann kann man sie **anonymisieren**: Alles, was auf die Person zurückführt, wird entfernt oder überschrieben, sodass niemand sie mehr zuordnen kann, auch das Studio nicht.

1. Welche Daten von Account 42 würdet ihr löschen, welche anonymisieren? Begründet.
2. Ist ein Account mit `email = 'geloescht-42@example.invalid'` und allen Logins mit Zeitpunkt, aber ohne IP-Adresse wirklich anonym? Was könnte jemand trotzdem noch herausfinden?
3. In den Backups von letzter Woche steht Account 42 noch drin. Was nun?

Ein Unternehmen hält solche Entscheidungen in einem **Löschkonzept** fest: welche Daten, wie lange, wie gelöscht.

*Antwort:* ...

# Ticket 8: „Alle Gilden sind weg!“

Bevor ihr das Unglück nachstellt, macht ihr ein **Backup**. Der SQL-Befehl `VACUUM INTO` schreibt eine vollständige, saubere Kopie der Datenbank in eine neue Datei.

Die Zelle führt ihn über eine eigene Verbindung aus. Ein Backup darf nicht laufen, während auf derselben Verbindung noch eine Abfrage offen ist, und das Notebook hält oft eine offen. Die Zieldatei darf noch nicht existieren, deshalb wird ein altes Backup vorher gelöscht.

In [ ]:
Path("backup.db").unlink(missing_ok=True)

con = sqlite3.connect(DB)
con.execute("VACUUM INTO 'backup.db'")
con.close()
print("Backup geschrieben:", Path("backup.db").stat().st_size // 1024, "KB")

Merkt euch den Stand vor dem Unglück:

In [ ]:
%%sql
SELECT (SELECT COUNT(*) FROM guild)             AS gilden,
       (SELECT COUNT(*) FROM player_guild)      AS mitgliedschaften,
       (SELECT COUNT(*) FROM player_guild_role) AS rollen;

### Übung 8.1
Jetzt das Unglück: Führt `DELETE FROM guild;` aus. Ohne `WHERE`. Zählt danach mit der Zelle oben noch einmal nach. Warum sind auch die Mitgliedschaften und Rollen weg?

*Antwort:* ...

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 8.2: Wiederherstellen
Mit `ATTACH` könnt ihr eine zweite Datenbankdatei unter einem Namen einbinden und ihre Tabellen mit `name.tabelle` ansprechen:

```sql
ATTACH DATABASE 'backup.db' AS sicherung;
SELECT COUNT(*) FROM sicherung.guild;
```

Holt die drei Tabellen aus dem Backup zurück (`INSERT INTO ... SELECT * FROM sicherung....`). Achtet auf die **Reihenfolge**: Was muss zuerst da sein, damit die Fremdschlüssel passen?

**Fertig, wenn** die Zählung wieder genau den Stand vor dem Unglück zeigt.

`ATTACH` gilt nur für die aktuelle Verbindung. Nach einem Restart ist das Backup wieder abgehängt.

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Merke: Backups

- **3-2-1-Regel:** mindestens **3** Kopien der Daten, auf **2** verschiedenen Speichermedien, **1** davon an einem anderen Ort.
- **Wiederherstellen üben.** Ein Backup, das nie zurückgespielt wurde, ist nur eine Hoffnung.
- **Wie viel Verlust ist erlaubt?** Wer einmal pro Nacht sichert, verliert im schlimmsten Fall einen ganzen Tag. Server-Datenbanken können zusätzlich jede Transaktion protokollieren und so auf einen beliebigen Zeitpunkt zurückspringen (Point-in-Time-Recovery).
- **Backups enthalten personenbezogene Daten.** Sie gehören ins Löschkonzept aus Ticket 7 und müssen genauso geschützt werden wie die Datenbank selbst.
- `DELETE` ohne `WHERE` passiert. Gegen solche Fehler hilft eine Transaktion: erst `BEGIN`, dann ändern, die Anzahl der geänderten Zeilen prüfen, dann `COMMIT` oder `ROLLBACK`.

# Was SQLite nicht kann

SQLite ist eine Datenbank **in einer Datei**, ohne eigenen Server. Für Lernen, Apps und kleine Programme ist das ideal. Große Mehrbenutzersysteme nutzen Server-Datenbanken wie PostgreSQL, MySQL/MariaDB, Microsoft SQL Server oder Oracle. Die können ein paar Dinge mehr:

| Funktion | Wozu | Beispiel (PostgreSQL) |
|---|---|---|
| Benutzer und Rechte | wer was darf (Ticket 4) | `GRANT SELECT ON support_account TO support;` |
| Stored Procedures | Programmlogik, die **in der Datenbank** gespeichert ist und dort läuft | siehe unten |
| viele gleichzeitige Schreibzugriffe | SQLite sperrt beim Schreiben die ganze Datei | |
| Netzwerkzugriff | viele Server greifen auf eine Datenbank zu | |

Eine **Stored Procedure** für den Handel aus Ticket 3:

```sql
-- PostgreSQL, funktioniert NICHT in SQLite
CREATE PROCEDURE gold_ueberweisen(von INTEGER, an INTEGER, betrag INTEGER)
LANGUAGE plpgsql
AS $$
BEGIN
    UPDATE player SET gold = gold - betrag WHERE id = von;
    UPDATE player SET gold = gold + betrag WHERE id = an;
END;
$$;

CALL gold_ueberweisen(15, 1, 500);
```

Die App ruft nur noch `CALL` auf und muss die einzelnen Schritte nicht kennen. Scheitert eine der beiden Anweisungen, wird der ganze `CALL` zurückgerollt.

### Und NoSQL?
Nicht jede Datenbank arbeitet mit Tabellen. Unter dem Begriff **NoSQL** fasst man andere Modelle zusammen:

| Art | Speichert | Beispiel | Typischer Einsatz |
|---|---|---|---|
| Dokumentdatenbank | JSON-ähnliche Dokumente | MongoDB | Daten mit wechselnder Struktur, z. B. Produktkataloge |
| Key-Value-Store | Schlüssel → Wert | Redis | Cache, Sitzungen, Ranglisten |
| Graphdatenbank | Knoten und Kanten | Neo4j | Netzwerke, z. B. „Freunde von Freunden“ |

Für Daten mit klaren Beziehungen und strengen Regeln (wie Gold, das nicht verschwinden darf) bleibt die relationale Datenbank meist die beste Wahl.

# Geschafft!

Ihr habt ein laufendes System durch acht Probleme gebracht. Weiter geht es mit [Teil 5: Eigene Erweiterung](../5-erweiterung/Anforderungen.md). Dort entscheidet ihr selbst, was das Spiel als Nächstes braucht.